# Adzuna Sector Job Pipeline — Full API Payload Notebook

This notebook fetches **full Adzuna API responses** for sector-level job searches.

It is designed for the workflow:

**8 sector keywords → Adzuna search → full raw payload storage → field flattening → sector hiring / noise summaries**

Compared with a minimal jobs extractor, this version keeps:
- the full raw JSON payload for each vacancy
- flattened API fields from the Adzuna response
- optional noise / quality signals that can be used later in scoring

The notebook is structured so you can run a small sample first, then scale to the full sector run.

In [1]:
import os
import json
import time
import sqlite3
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm.auto import tqdm


In [2]:
# =========================================================
# Configuration
# =========================================================
INPUT_PATH = Path("../output/UKcompanies_8_sectors_cleaned.csv")

OUTPUT_DIR = Path("../output/adzuna_sector_fullfield_output")
OUTPUT_DIR.mkdir(exist_ok=True)

RAW_JOBS_CSV = OUTPUT_DIR / "adzuna_sector_jobs_raw_full.csv"
SECTOR_SUMMARY_CSV = OUTPUT_DIR / "adzuna_sector_hiring_summary.csv"
CACHE_DB = OUTPUT_DIR / "adzuna_sector_cache.sqlite"

# You can set these in your environment instead of hardcoding them.
# Example:
#   export ADZUNA_APP_ID="..."
#   export ADZUNA_APP_KEY="..."
ADZUNA_APP_ID = os.getenv("ADZUNA_APP_ID", "9d40f51d").strip()
ADZUNA_APP_KEY = os.getenv("ADZUNA_APP_KEY", "6a151e5f23976ded9a47ce1cf059df04").strip()

if not ADZUNA_APP_ID or not ADZUNA_APP_KEY:
    raise ValueError("Please set ADZUNA_APP_ID and ADZUNA_APP_KEY in your environment variables.")

BASE_URL = "https://api.adzuna.com/v1/api/jobs"
COUNTRY = "gb"
WHERE_VALUE = "uk"

RESULTS_PER_PAGE = 50
MAX_PAGES_PER_QUERY = 2
REQUEST_SLEEP_SECONDS = 0.4

# Run a smaller sample first, then set this to None or a larger number.
SAMPLE_SIZE = 50

# Sector keyword map used for sector-level hiring estimation.
SECTOR_QUERY_MAP = {
    "Manufacturing": [
        "manufacturing",
        "production",
        "factory",
        "process engineer",
        "maintenance engineer",
    ],
    "Healthcare": [
        "healthcare",
        "nurse",
        "medical",
        "care worker",
        "clinical",
    ],
    "Technology, legal & professional": [
        "software engineer",
        "developer",
        "data analyst",
        "consultant",
        "solicitor",
    ],
    "Agriculture": [
        "agriculture",
        "farm",
        "farmer",
        "horticulture",
        "agronomist",
    ],
    "Real Estate": [
        "property",
        "estate agent",
        "lettings",
        "property manager",
        "surveying",
    ],
    "Wholesale & Retail": [
        "retail",
        "sales assistant",
        "store manager",
        "wholesale",
        "merchandiser",
    ],
    "Public sector, education & charities": [
        "teacher",
        "lecturer",
        "council",
        "charity",
        "fundraising",
    ],
    "Fast growth & emerging sector": [
        "product manager",
        "data scientist",
        "machine learning",
        "growth marketing",
        "startup",
    ],
}

TARGET_SECTORS = list(SECTOR_QUERY_MAP.keys())


In [3]:
# =========================================================
# Load Companies House company data
# =========================================================
df = pd.read_csv(INPUT_PATH, dtype=str)

required_cols = {"CompanyNumber", "CompanyName", "primary_sector"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"Input file missing required columns: {sorted(missing)}")

df["primary_sector"] = df["primary_sector"].astype("string").str.strip()

focus_companies = df[df["primary_sector"].isin(TARGET_SECTORS)].copy()
focus_companies = (
    focus_companies[["CompanyNumber", "CompanyName", "primary_sector"]]
    .drop_duplicates()
    .head(SAMPLE_SIZE)
    .copy()
)

focus_companies.shape


(50, 3)

In [4]:
# =========================================================
# Requests session with retry support
# =========================================================
def make_session() -> requests.Session:
    session = requests.Session()
    retry = Retry(
        total=5,
        read=5,
        connect=5,
        backoff_factor=1.5,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=frozenset(["GET"]),
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session

session = make_session()


In [5]:
# =========================================================
# SQLite cache helpers
# =========================================================
def init_cache():
    conn = sqlite3.connect(CACHE_DB)
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE IF NOT EXISTS sector_cache (
            cache_key TEXT PRIMARY KEY,
            response_json TEXT,
            fetched_at TEXT
        )
    """)
    conn.commit()
    conn.close()

def sector_cache_key(sector_name: str, query: str, where: Optional[str], page: int) -> str:
    return f"{sector_name}|{query.lower().strip()}|{(where or '').lower().strip()}|{page}"

def cache_get(key: str) -> Optional[dict]:
    conn = sqlite3.connect(CACHE_DB)
    cur = conn.cursor()
    cur.execute("SELECT response_json FROM sector_cache WHERE cache_key = ?", (key,))
    row = cur.fetchone()
    conn.close()
    if not row:
        return None
    try:
        return json.loads(row[0])
    except Exception:
        return None

def cache_put(key: str, response: dict):
    conn = sqlite3.connect(CACHE_DB)
    cur = conn.cursor()
    cur.execute("""
        INSERT OR REPLACE INTO sector_cache (cache_key, response_json, fetched_at)
        VALUES (?, ?, datetime('now'))
    """, (key, json.dumps(response, ensure_ascii=False)))
    conn.commit()
    conn.close()

init_cache()
print("Cache initialized.")


Cache initialized.


In [6]:
# =========================================================
# Adzuna API request
# =========================================================
def adzuna_search_jobs(query: str, where: Optional[str] = None, page: int = 1, results_per_page: int = 50) -> dict:
    """
    Query the Adzuna API for job vacancies.
    """
    url = f"{BASE_URL}/{COUNTRY}/search/{page}"
    params = {
        "app_id": ADZUNA_APP_ID,
        "app_key": ADZUNA_APP_KEY,
        "what": query,
        "results_per_page": results_per_page,
        "content-type": "application/json",
    }
    if where:
        params["where"] = where

    resp = session.get(url, params=params, timeout=40)
    resp.raise_for_status()
    return resp.json()


In [7]:
# =========================================================
# Full payload parser + full-field extraction
# =========================================================
AGENCY_HINTS = [
    "recruitment", "recruiter", "hays", "reed", "adecco", "manpower",
    "michael page", "page personnel", "randstad", "robert walters",
    "spire", "morson", "cvlibrary", "technical recruiter"
]

GENERIC_TITLE_HINTS = [
    "assistant", "administrator", "general manager", "manager",
    "operational", "support", "operator", "assistant manager"
]

def _contains_any(text: str, keywords: list[str]) -> bool:
    text = (text or "").lower()
    return any(k.lower() in text for k in keywords)

def parse_jobs_full(data: dict, sector: str, keyword: str, page: int) -> list[dict]:
    """
    Preserve the full raw Adzuna payload while also extracting flattened fields.
    """
    rows = []

    for item in data.get("results", []):
        # Flatten nested JSON while keeping all available API keys.
        flat = pd.json_normalize(item, sep="__").to_dict(orient="records")[0]

        loc = item.get("location") or {}
        area = loc.get("area") or []

        title = str(flat.get("title", "") or "")
        desc = str(flat.get("description", "") or "")
        company_display = str((item.get("company") or {}).get("display_name", "") or "")

        salary_min = flat.get("salary_min")
        salary_max = flat.get("salary_max")
        salary_missing = pd.isna(salary_min) and pd.isna(salary_max)

        row = {
            # query metadata
            "sector": sector,
            "keyword": keyword,
            "page": page,
            "query_sector_keyword": f"{sector}::{keyword}",

            # full flattened raw payload
            **flat,

            # explicit convenience fields
            "company_display_name": company_display,
            "location_display_name": loc.get("display_name", ""),
            "location_area": " > ".join([x for x in area if x]),
            "raw_item_json": json.dumps(item, ensure_ascii=False, default=str),

            # derived quality / noise signals
            "title_len": len(title.strip()),
            "desc_len": len(desc.strip()),
            "has_salary": not salary_missing,
            "salary_missing": salary_missing,
            "location_depth": len(area),
            "possible_agency_noise": _contains_any(company_display, AGENCY_HINTS),
            "generic_title_noise": _contains_any(title, GENERIC_TITLE_HINTS),
            "keyword_in_title": keyword.lower() in title.lower(),
            "keyword_in_desc": keyword.lower() in desc.lower(),
        }

        rows.append(row)

    return rows


In [8]:
# =========================================================
# Fetch a single sector keyword across pages
# =========================================================
def fetch_jobs_for_sector(
    sector_name: str,
    query: str,
    where: str = WHERE_VALUE,
    max_pages: int = MAX_PAGES_PER_QUERY,
    use_cache: bool = True,
) -> pd.DataFrame:
    rows = []

    for page in range(1, max_pages + 1):
        key = sector_cache_key(sector_name, query, where, page)
        data = cache_get(key) if use_cache else None

        if data is None:
            try:
                data = adzuna_search_jobs(
                    query=query,
                    where=where,
                    page=page,
                    results_per_page=RESULTS_PER_PAGE,
                )
                if use_cache:
                    cache_put(key, data)
            except Exception as e:
                rows.append({
                    "sector": sector_name,
                    "keyword": query,
                    "page": page,
                    "error": str(e),
                })
                break

            time.sleep(REQUEST_SLEEP_SECONDS)

        results = data.get("results", []) if isinstance(data, dict) else []
        total = data.get("count", None) if isinstance(data, dict) else None

        for item in results:
            # Parse and preserve all raw payload fields
            parsed_rows = parse_jobs_full(data={"results": [item]}, sector=sector_name, keyword=query, page=page)
            if parsed_rows:
                parsed_rows[0]["adzuna_total"] = total
                rows.append(parsed_rows[0])

        if len(results) < RESULTS_PER_PAGE:
            break

    return pd.DataFrame(rows)


In [9]:
# =========================================================
# Batch run across all sectors / keywords
# =========================================================
def run_sector_batch(sector_query_map: dict, where: str = WHERE_VALUE) -> pd.DataFrame:
    all_raw = []

    pairs = [(sector, kw) for sector, kws in sector_query_map.items() for kw in kws]
    for sector_name, keyword in tqdm(pairs, desc="Sector keyword queries"):
        df_q = fetch_jobs_for_sector(
            sector_name=sector_name,
            query=keyword,
            where=where,
            max_pages=MAX_PAGES_PER_QUERY,
            use_cache=True,
        )

        if not df_q.empty:
            all_raw.append(df_q)

        time.sleep(REQUEST_SLEEP_SECONDS)

    raw_df = pd.concat(all_raw, ignore_index=True) if all_raw else pd.DataFrame()
    if not raw_df.empty:
        raw_df.to_csv(RAW_JOBS_CSV, index=False, encoding="utf-8-sig")

    return raw_df


In [10]:
# =========================================================
# Run a sample first
# =========================================================
sector_raw = run_sector_batch(SECTOR_QUERY_MAP, where=WHERE_VALUE)
print("sector_raw shape:", sector_raw.shape)
sector_raw.head(20)


Sector keyword queries:   0%|          | 0/40 [00:00<?, ?it/s]

sector_raw shape: (3932, 40)


,sector,keyword,page,query_sector_keyword,contract_type,latitude,salary_max,adref,__CLASS__,description,...,desc_len,has_salary,salary_missing,location_depth,possible_agency_noise,generic_title_noise,keyword_in_title,keyword_in_desc,adzuna_total,contract_time
0,Manufacturing,manufacturing,1,Manufacturing::manufacturing,permanent,51.080127,49836.43,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTc2MjcxNjkwMCIsI...,Adzuna::API::Response::Job,"Management Accountant - Manufacturing, Permane...",...,500,True,False,3,True,False,True,True,53539,NaN
1,Manufacturing,manufacturing,1,Manufacturing::manufacturing,NaN,53.757702,49556.15,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiMkxmYXZ4MXY4UkdGa...,Adzuna::API::Response::Job,"Financial Controller job, Manufacturing sector...",...,500,True,False,4,True,False,True,True,53539,NaN
2,Manufacturing,manufacturing,1,Manufacturing::manufacturing,NaN,53.480701,45532.82,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTcyNDgxNTI4MiIsI...,Adzuna::API::Response::Job,Work in a broad role in a growing business You...,...,500,True,False,4,True,True,True,True,53539,NaN
3,Manufacturing,manufacturing,1,Manufacturing::manufacturing,NaN,53.754200,51775.75,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiMkxmYXZ4MXY4UkdGa...,Adzuna::API::Response::Job,Job Title: Manufacturing Engineer – Electrical...,...,500,True,False,5,False,False,True,True,53539,NaN
4,Manufacturing,manufacturing,1,Manufacturing::manufacturing,NaN,53.758495,43891.34,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiMkxmYXZ4MXY4UkdGa...,Adzuna::API::Response::Job,Job Title: Manufacturing Engineer – Electrical...,...,500,True,False,5,False,False,True,True,53539,NaN
5,Manufacturing,manufacturing,1,Manufacturing::manufacturing,NaN,53.798801,44280.07,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiMkxmYXZ4MXY4UkdGa...,Adzuna::API::Response::Job,Job Title: Manufacturing Engineer – Electrical...,...,500,True,False,5,False,False,True,True,53539,NaN
6,Manufacturing,manufacturing,1,Manufacturing::manufacturing,contract,51.620399,47659.41,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTcwNzMzMTA5NyIsI...,Adzuna::API::Response::Job,12 month contract for Manufacturing Accountant...,...,500,True,False,3,True,False,True,True,53539,NaN
7,Manufacturing,manufacturing,1,Manufacturing::manufacturing,permanent,52.647999,55000.00,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTc1MTI2OTg5NyIsI...,Adzuna::API::Response::Job,Manufacturing Engineer A recognised high volum...,...,500,True,False,4,False,False,True,True,53539,NaN
8,Manufacturing,manufacturing,1,Manufacturing::manufacturing,permanent,NaN,44273.44,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTc0NDQ4NjM2OCIsI...,Adzuna::API::Response::Job,This is an excellent opportunity for a Estimat...,...,500,True,False,3,False,False,True,True,53539,full_time
9,Manufacturing,manufacturing,1,Manufacturing::manufacturing,permanent,51.403702,35000.00,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiMkxmYXZ4MXY4UkdGa...,Adzuna::API::Response::Job,"Location: Thatcham, Berkshire Department: Manu...",...,500,True,False,4,False,False,True,True,53539,full_time


In [20]:
# =========================================================
# Build sector summary with noise / quality metrics
# job_id / id compatible version
# =========================================================
def build_sector_hiring_summary(raw_df: pd.DataFrame) -> pd.DataFrame:
    if raw_df.empty:
        return pd.DataFrame()

    out = raw_df.copy()

    # -------------------------------------------------
    # 1) Normalize job identifier
    # Adzuna raw payload usually uses `id`, not `job_id`
    # -------------------------------------------------
    if "job_id" not in out.columns:
        if "id" in out.columns:
            out["job_id"] = out["id"]
        else:
            out["job_id"] = pd.NA

    # -------------------------------------------------
    # 2) Deduplicate within sector
    # -------------------------------------------------
    if "sector" in out.columns:
        out = out.drop_duplicates(subset=["sector", "job_id"]).copy()
    else:
        raise KeyError("Expected a 'sector' column in raw_df.")

    # -------------------------------------------------
    # 3) Robust datetime parsing
    # Use timezone-naive datetimes consistently
    # -------------------------------------------------
    def _to_naive_datetime(series: pd.Series) -> pd.Series:
        dt = pd.to_datetime(series, errors="coerce")
        try:
            if hasattr(dt.dt, "tz") and dt.dt.tz is not None:
                dt = dt.dt.tz_localize(None)
        except Exception:
            pass
        return dt

    if "created" in out.columns:
        out["created_dt"] = _to_naive_datetime(out["created"])
    else:
        out["created_dt"] = pd.NaT

    if "modified" in out.columns:
        out["modified_dt"] = _to_naive_datetime(out["modified"])
    else:
        out["modified_dt"] = pd.NaT

    now = pd.Timestamp.now()

    # -------------------------------------------------
    # 4) Safe time-delta features
    # -------------------------------------------------
    out["days_since_created"] = (now - out["created_dt"]).dt.days
    out["days_since_modified"] = (now - out["modified_dt"]).dt.days

    # -------------------------------------------------
    # 5) Sector aggregation
    # -------------------------------------------------
    agg_dict = {
        "raw_rows": ("job_id", "size"),
        "unique_jobs": ("job_id", "nunique"),
        "vacancy_count": ("job_id", "nunique"),
        "unique_titles": ("title", "nunique") if "title" in out.columns else ("job_id", "nunique"),
        "unique_locations": ("location_display_name", "nunique") if "location_display_name" in out.columns else ("job_id", "nunique"),
        "latest_job_update": ("modified_dt", "max"),
        "earliest_job_post": ("created_dt", "min"),
        "query_coverage": ("keyword", "nunique") if "keyword" in out.columns else ("job_id", "nunique"),
    }

    # Optional noise / quality features
    if "salary_missing" in out.columns:
        agg_dict["salary_missing_ratio"] = ("salary_missing", "mean")
    else:
        out["salary_missing"] = False
        agg_dict["salary_missing_ratio"] = ("salary_missing", "mean")

    if "possible_agency_noise" in out.columns:
        agg_dict["agency_noise_ratio"] = ("possible_agency_noise", "mean")
    else:
        out["possible_agency_noise"] = False
        agg_dict["agency_noise_ratio"] = ("possible_agency_noise", "mean")

    if "generic_title_noise" in out.columns:
        agg_dict["generic_title_noise_ratio"] = ("generic_title_noise", "mean")
    else:
        out["generic_title_noise"] = False
        agg_dict["generic_title_noise_ratio"] = ("generic_title_noise", "mean")

    if "desc_len" in out.columns:
        agg_dict["short_desc_ratio"] = ("desc_len", lambda s: (s < 120).mean())
    else:
        out["desc_len"] = np.nan
        agg_dict["short_desc_ratio"] = ("desc_len", lambda s: (s < 120).mean())

    summary = (
        out.groupby("sector", dropna=False)
        .agg(**agg_dict)
        .reset_index()
    )

    # -------------------------------------------------
    # 6) Overlap / duplication
    # -------------------------------------------------
    summary["duplicate_overlap_ratio"] = (
        1 - summary["unique_jobs"] / summary["raw_rows"].clip(lower=1)
    )

    # -------------------------------------------------
    # 7) Normalize positive features
    # -------------------------------------------------
    max_vacancy = max(summary["vacancy_count"].max(), 1)
    max_query_coverage = max(summary["query_coverage"].max(), 1)
    max_titles = max(summary["unique_titles"].max(), 1)
    max_locations = max(summary["unique_locations"].max(), 1)

    summary["volume_norm"] = summary["vacancy_count"] / max_vacancy
    summary["coverage_norm"] = summary["query_coverage"] / max_query_coverage
    summary["title_breadth_norm"] = summary["unique_titles"] / max_titles
    summary["location_breadth_norm"] = summary["unique_locations"] / max_locations
    summary["recency_norm"] = np.clip(
        1 - (summary["earliest_job_post"].notna().astype(int) * 0 + summary["earliest_job_post"].isna().astype(int) * 0 + (now - summary["earliest_job_post"]).dt.days.fillna(999) / 180),
        0,
        1
    )

    # 更稳妥的 recency：用 latest_job_update 来算
    summary["recency_norm"] = np.clip(
        1 - ((now - summary["latest_job_update"]).dt.days.fillna(999) / 180),
        0,
        1
    )

    # -------------------------------------------------
    # 8) Sector hiring score
    # -------------------------------------------------
    summary["sector_hiring_score"] = (
        100 * (
            0.35 * summary["volume_norm"] +
            0.20 * summary["recency_norm"] +
            0.20 * summary["coverage_norm"] +
            0.15 * summary["title_breadth_norm"] +
            0.10 * summary["location_breadth_norm"]
        )
    ).round(1)

    # -------------------------------------------------
    # 9) Noise score
    # -------------------------------------------------
    summary["noise_score"] = (
        100 * (
            0.30 * summary["salary_missing_ratio"].fillna(0) +
            0.25 * summary["duplicate_overlap_ratio"].fillna(0) +
            0.20 * summary["agency_noise_ratio"].fillna(0) +
            0.15 * summary["generic_title_noise_ratio"].fillna(0) +
            0.10 * summary["short_desc_ratio"].fillna(0)
        )
    ).round(1)

    return summary

In [21]:
sector_summary = build_sector_hiring_summary(sector_raw)
sector_summary.to_csv(SECTOR_SUMMARY_CSV, index=False, encoding="utf-8-sig")
sector_summary.head()


,sector,raw_rows,unique_jobs,vacancy_count,unique_titles,unique_locations,latest_job_update,earliest_job_post,query_coverage,salary_missing_ratio,...,generic_title_noise_ratio,short_desc_ratio,duplicate_overlap_ratio,volume_norm,coverage_norm,title_breadth_norm,location_breadth_norm,recency_norm,sector_hiring_score,noise_score
0,Agriculture,417,417,417,297,255,NaT,2023-02-01 19:35:49,5,0.000000,...,0.275779,0.0,0.0,0.834,1.0,1.000000,0.761194,0.0,71.8,8.0
1,Fast growth & emerging sector,499,499,499,253,162,NaT,2023-12-28 17:21:06,5,0.002004,...,0.348697,0.0,0.0,0.998,1.0,0.851852,0.483582,0.0,72.5,8.2
2,Healthcare,500,500,500,173,335,NaT,2025-08-10 07:34:41,5,0.000000,...,0.422000,0.0,0.0,1.000,1.0,0.582492,1.000000,0.0,73.7,11.0
3,Manufacturing,499,499,499,133,319,NaT,2024-07-31 15:02:09,5,0.000000,...,0.124248,0.0,0.0,0.998,1.0,0.447811,0.952239,0.0,71.2,6.4
4,"Public sector, education & charities",472,472,472,296,258,NaT,2025-01-16 12:25:40,5,0.000000,...,0.095339,0.0,0.0,0.944,1.0,0.996633,0.770149,0.0,75.7,4.2


In [22]:
sector_raw

,sector,keyword,page,query_sector_keyword,contract_type,latitude,salary_max,adref,__CLASS__,description,...,desc_len,has_salary,salary_missing,location_depth,possible_agency_noise,generic_title_noise,keyword_in_title,keyword_in_desc,adzuna_total,contract_time
0,Manufacturing,manufacturing,1,Manufacturing::manufacturing,permanent,51.080127,49836.43,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTc2MjcxNjkwMCIsI...,Adzuna::API::Response::Job,"Management Accountant - Manufacturing, Permane...",...,500,True,False,3,True,False,True,True,53539,NaN
1,Manufacturing,manufacturing,1,Manufacturing::manufacturing,NaN,53.757702,49556.15,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiMkxmYXZ4MXY4UkdGa...,Adzuna::API::Response::Job,"Financial Controller job, Manufacturing sector...",...,500,True,False,4,True,False,True,True,53539,NaN
2,Manufacturing,manufacturing,1,Manufacturing::manufacturing,NaN,53.480701,45532.82,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTcyNDgxNTI4MiIsI...,Adzuna::API::Response::Job,Work in a broad role in a growing business You...,...,500,True,False,4,True,True,True,True,53539,NaN
3,Manufacturing,manufacturing,1,Manufacturing::manufacturing,NaN,53.754200,51775.75,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiMkxmYXZ4MXY4UkdGa...,Adzuna::API::Response::Job,Job Title: Manufacturing Engineer – Electrical...,...,500,True,False,5,False,False,True,True,53539,NaN
4,Manufacturing,manufacturing,1,Manufacturing::manufacturing,NaN,53.758495,43891.34,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiMkxmYXZ4MXY4UkdGa...,Adzuna::API::Response::Job,Job Title: Manufacturing Engineer – Electrical...,...,500,True,False,5,False,False,True,True,53539,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3927,Fast growth & emerging sector,startup,2,Fast growth & emerging sector::startup,permanent,54.448101,30000.00,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiUUtEekNoNXY4Ukd3c...,Adzuna::API::Response::Job,Permanent Nightshift Production Operativerequi...,...,500,True,False,3,True,False,False,False,4037,NaN
3928,Fast growth & emerging sector,startup,2,Fast growth & emerging sector::startup,permanent,NaN,32280.03,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiUUtEekNoNXY4Ukd3c...,Adzuna::API::Response::Job,Your new company Join a growing and supportive...,...,500,True,False,1,True,False,False,True,4037,NaN
3929,Fast growth & emerging sector,startup,2,Fast growth & emerging sector::startup,permanent,NaN,100000.00,eyJhbGciOiJIUzI1NiJ9.eyJzIjoiUUtEekNoNXY4Ukd3c...,Adzuna::API::Response::Job,"Software Engineer - Up to £160,000 - London Ti...",...,500,True,False,2,True,False,False,True,4037,NaN
3930,Fast growth & emerging sector,startup,2,Fast growth & emerging sector::startup,NaN,NaN,30000.00,eyJhbGciOiJIUzI1NiJ9.eyJpIjoiNTc2MTU1MTAxOSIsI...,Adzuna::API::Response::Job,Associate Recruitment Consultant | Belfast Cit...,...,500,True,False,3,False,False,False,True,4037,NaN


In [23]:
sector_raw.columns

Index(['sector', 'keyword', 'page', 'query_sector_keyword', 'contract_type',
       'latitude', 'salary_max', 'adref', '__CLASS__', 'description',
       'salary_is_predicted', 'longitude', 'redirect_url', 'salary_min', 'id',
       'created', 'title', 'category__label', 'category__tag',
       'category____CLASS__', 'location__area', 'location____CLASS__',
       'location__display_name', 'company__display_name', 'company____CLASS__',
       'company_display_name', 'location_display_name', 'location_area',
       'raw_item_json', 'title_len', 'desc_len', 'has_salary',
       'salary_missing', 'location_depth', 'possible_agency_noise',
       'generic_title_noise', 'keyword_in_title', 'keyword_in_desc',
       'adzuna_total', 'contract_time'],
      dtype='object')

In [24]:
# =========================================================
# Merge sector score back to company table
# =========================================================
company_enriched = df.merge(
    sector_summary[
        [
            "sector",
            "vacancy_count",
            "unique_jobs",
            "unique_titles",
            "unique_locations",
            "query_coverage",
            "latest_job_update",
            "earliest_job_post",
            "days_since_created",
            "sector_hiring_score",
            "noise_score",
        ]
    ],
    left_on="primary_sector",
    right_on="sector",
    how="left"
)

company_enriched = company_enriched.drop(columns=["sector"])
company_enriched["hiring_score"] = company_enriched["sector_hiring_score"]

company_enriched.head(20)


KeyError: "['days_since_created'] not in index"

In [ ]:
# =========================================================
# Export company-level output
# =========================================================
company_enriched.to_csv(
    OUTPUT_DIR / "companies_with_sector_hiring_full_payload.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Saved:")
print(" -", RAW_JOBS_CSV)
print(" -", SECTOR_SUMMARY_CSV)
print(" -", OUTPUT_DIR / "companies_with_sector_hiring_full_payload.csv")


In [ ]:
# =========================================================
# Optional full run
# =========================================================
# If the sample looks good, run on the full dataset:
#
# focus_companies_full = df[df["primary_sector"].isin(TARGET_SECTORS)].copy()
# sector_raw_full = run_sector_batch(SECTOR_QUERY_MAP, where=WHERE_VALUE)
# sector_summary_full = build_sector_hiring_summary(sector_raw_full)
# sector_summary_full.to_csv(SECTOR_SUMMARY_CSV, index=False, encoding="utf-8-sig")
# company_enriched_full = df.merge(
#     sector_summary_full[
#         [
#             "sector",
#             "vacancy_count",
#             "unique_jobs",
#             "unique_titles",
#             "unique_locations",
#             "query_coverage",
#             "latest_job_update",
#             "earliest_job_post",
#             "days_since_created",
#             "sector_hiring_score",
#             "noise_score",
#         ]
#     ],
#     left_on="primary_sector",
#     right_on="sector",
#     how="left"
# ).drop(columns=["sector"])
# company_enriched_full["hiring_score"] = company_enriched_full["sector_hiring_score"]
# company_enriched_full.to_csv(
#     OUTPUT_DIR / "companies_with_sector_hiring_full_payload_fullrun.csv",
#     index=False,
#     encoding="utf-8-sig"
# )


## Notes

- This notebook preserves the **full Adzuna raw payload** for each vacancy using `raw_item_json` and the flattened API fields.
- It also computes sector-level hiring and noise summaries so you can later build a more robust scoring model.
- The first run uses a sample size to validate the pipeline. Increase `SAMPLE_SIZE` or run the full block when ready.